# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

- Identify adaptations in mouse reward circuits associated with oxycodone physical dependence
- Reveal that chronic neuropathic pain states affect sensory, effective and molecular signatures of opioid withdrawl
- Demonstrate that treatment with HDAC1/HDAC2 inhibitor, Regenacy Brain Class 1 HDAC Inhibitor (RBC1HI), alleviates mechanical allodynia and prevents the expression of sensory and affective manifestations of oxycodone withdrawl, pointing to a new therapeutic approach for managment of chronic pain in opioid-dependent individuals.

What do the conditions mean?

**oxy**: reception of chronic oxycodone injections (treated group)


**sal**: saline injections (control group)

What do the genotypes mean?

**SNI**: spared nerve injury


**Sham**: control group

**Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.**

**What would you do?**

Frist of all, I would normalize the data that they hand out to me (for example do the log of every data that I have). Ideally, each data point would be labeled with some kind of ID, so I could differenciate from which condition each group is. Finally, I would compare the differential expression of each gene from each condition. 
- quality control, normal, differential expression

**Which groups would you compare to each other?**

It would be always treatment vs control. Therefore, if I am comparing the areas of the brain where genes might be differentially expressed depending on the condition, I would compare: NAc-SNI vs NAc-Sham, mPFC-SNI vs mPFC-Sham and VTA-SNI vs VTA-Sham. And, if we are comparing cell types it would be: cell type-SNI vs cell type-Sham.

As well, we would have to take into consideration if they took the chronic oxycodone injection and the saline injection. 

**Please also mention which outcome you would expect to see from each comparison.**

I would expect that genes might be significantly expressed in one of the conditions. 

Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [1]:
import os
print(os.getcwd())

/home/paula/.ssh/computational-workflows-2026/day_02


In [1]:
import pandas as pd
import openpyxl

In [8]:
data = pd.read_excel("conditions_runs_oxy_project.xlsx")

data

,Patient,Run,RNA-seq,DNA-seq,condition: Sal,Condition: Oxy,Genotype: SNI,Genotype: Sham
0,?,SRR23195505,x,NaN,x,NaN,x,NaN
1,?,SRR23195506,x,NaN,NaN,x,NaN,x
2,?,SRR23195507,x,NaN,x,NaN,NaN,x
3,?,SRR23195508,x,NaN,NaN,x,x,NaN
4,?,SRR23195509,x,NaN,NaN,x,x,NaN
5,?,SRR23195510,x,NaN,x,NaN,x,NaN
6,?,SRR23195511,x,NaN,NaN,x,NaN,x
7,?,SRR23195512,x,NaN,x,NaN,NaN,x
8,?,SRR23195513,x,NaN,x,NaN,x,NaN
9,?,SRR23195514,x,NaN,NaN,x,NaN,x


**1. How many samples do you have per condition?**

There are a total of 8 measurments per condition

**2. How many samples do you have per genotype?**

There are a total of 8 samples per genotype (16 from Sal condition and 16 from Oxy condition)

**3. How often do you have each condition per genotype?**

8

They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [13]:
space = pd.read_csv("base_counts.csv")

space

,Run,Bases
0,SRR23195505,6922564500
1,SRR23195506,7859530800
2,SRR23195507,8063298900
3,SRR23195508,6927786900
4,SRR23195509,7003550100
5,SRR23195510,7377388500
6,SRR23195511,6456390900
7,SRR23195512,7462857900
8,SRR23195513,8099181600
9,SRR23195514,7226808600


In [16]:
data = data.merge(space, on="Run", how="left")
data

,Patient,Run,RNA-seq,DNA-seq,condition: Sal,Condition: Oxy,Genotype: SNI,Genotype: Sham,Bases_x,Bases_y
0,?,SRR23195505,x,NaN,x,NaN,x,NaN,6922564500,6922564500
1,?,SRR23195506,x,NaN,NaN,x,NaN,x,7859530800,7859530800
2,?,SRR23195507,x,NaN,x,NaN,NaN,x,8063298900,8063298900
3,?,SRR23195508,x,NaN,NaN,x,x,NaN,6927786900,6927786900
4,?,SRR23195509,x,NaN,NaN,x,x,NaN,7003550100,7003550100
5,?,SRR23195510,x,NaN,x,NaN,x,NaN,7377388500,7377388500
6,?,SRR23195511,x,NaN,NaN,x,NaN,x,6456390900,6456390900
7,?,SRR23195512,x,NaN,x,NaN,NaN,x,7462857900,7462857900
8,?,SRR23195513,x,NaN,x,NaN,x,NaN,8099181600,8099181600
9,?,SRR23195514,x,NaN,NaN,x,NaN,x,7226808600,7226808600


In [20]:
data = data.sort_values(by=["Bases_x"])
data

,Patient,Run,RNA-seq,DNA-seq,condition: Sal,Condition: Oxy,Genotype: SNI,Genotype: Sham,Bases_x,Bases_y
11,?,SRR23195516,x,NaN,NaN,x,x,NaN,6203117700,6203117700
6,?,SRR23195511,x,NaN,NaN,x,NaN,x,6456390900,6456390900
12,?,SRR23195517,x,NaN,NaN,x,x,NaN,6863840400,6863840400
0,?,SRR23195505,x,NaN,x,NaN,x,NaN,6922564500,6922564500
3,?,SRR23195508,x,NaN,NaN,x,x,NaN,6927786900,6927786900
14,?,SRR23195519,x,NaN,NaN,x,NaN,x,6996050100,6996050100
4,?,SRR23195509,x,NaN,NaN,x,x,NaN,7003550100,7003550100
9,?,SRR23195514,x,NaN,NaN,x,NaN,x,7226808600,7226808600
5,?,SRR23195510,x,NaN,x,NaN,x,NaN,7377388500,7377388500
7,?,SRR23195512,x,NaN,x,NaN,NaN,x,7462857900,7462857900


In [21]:
smallest_runs = data.nsmallest(2, "Bases_x")
smallest_runs

,Patient,Run,RNA-seq,DNA-seq,condition: Sal,Condition: Oxy,Genotype: SNI,Genotype: Sham,Bases_x,Bases_y
11,?,SRR23195516,x,NaN,NaN,x,x,NaN,6203117700,6203117700
6,?,SRR23195511,x,NaN,NaN,x,NaN,x,6456390900,6456390900


While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.

In [ ]:
!nextflow run nf-core/fetchngs -r 1.13.0 -profile docker --input smallest_runs.csv --outdir fetchengs_results -c /home/paula/.ssh/computational-workflows-2026/nextflow.config


 N E X T F L O W   ~  version 26.04.6

Pulling nf-core/fetchngs:1.13.0 ...
 downloaded from https://github.com/nf-core/fetchngs.git
Launching `https://github.com/nf-core/fetchngs` [mighty_nightingale] revision: 955c892d80 [1.13.0]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/fetchngs 1.13.0
------------------------------------------------------

Input/output options
  input              : smallest_runs.csv
  outdir             : fetchengs_results

Deprecated options
  trace_report_suffix: 2026-09-29_11-30-44

Core Nextflow options
  revision           : 1.13.0
  runName            : mighty_nightingale
  containerEngine    : docker
  launchDir          : /home/paula/.ssh/c